In [3]:
import argparse
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics import pairwise_distances


# --------------------------------------------------
# Disjoint Set / Union-Find for Kruskal's Algorithm
# --------------------------------------------------
class DisjointSet:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, x, y):
        rx = self.find(x)
        ry = self.find(y)

        if rx == ry:
            return False

        if self.rank[rx] < self.rank[ry]:
            self.parent[rx] = ry
        elif self.rank[rx] > self.rank[ry]:
            self.parent[ry] = rx
        else:
            self.parent[ry] = rx
            self.rank[rx] += 1

        return True


# --------------------------------------------------
# Build complete weighted graph from pairwise distances
# --------------------------------------------------
def build_complete_graph(X, metric="euclidean"):
    dist_matrix = pairwise_distances(X, metric=metric)
    n = len(X)
    edges = []

    for i in range(n):
        for j in range(i + 1, n):
            edges.append((i, j, dist_matrix[i, j]))

    return edges, dist_matrix


# --------------------------------------------------
# Kruskal's algorithm for MST
# --------------------------------------------------
def kruskal_mst(n, edges):
    edges_sorted = sorted(edges, key=lambda x: x[2])
    ds = DisjointSet(n)
    mst_edges = []

    for u, v, w in edges_sorted:
        if ds.union(u, v):
            mst_edges.append((u, v, w))
            if len(mst_edges) == n - 1:
                break

    return mst_edges


# --------------------------------------------------
# Form clusters by removing (k-1) largest MST edges
# --------------------------------------------------
def mst_clustering(n, mst_edges, k):
    if k <= 0 or k > n:
        raise ValueError("k must be between 1 and number of samples.")

    # Sort MST edges in descending order by weight
    edges_desc = sorted(mst_edges, key=lambda x: x[2], reverse=True)

    # Remove the (k-1) most expensive edges
    edges_to_keep = edges_desc[k - 1:] if k > 1 else edges_desc

    # Create adjacency list for remaining forest
    adj = [[] for _ in range(n)]
    for u, v, w in edges_to_keep:
        adj[u].append(v)
        adj[v].append(u)

    # Find connected components = clusters
    labels = [-1] * n
    cluster_id = 0

    for i in range(n):
        if labels[i] == -1:
            stack = [i]
            labels[i] = cluster_id

            while stack:
                node = stack.pop()
                for nei in adj[node]:
                    if labels[nei] == -1:
                        labels[nei] = cluster_id
                        stack.append(nei)

            cluster_id += 1

    return np.array(labels)


# --------------------------------------------------
# Relabel clusters to 0,1,2,... cleanly
# --------------------------------------------------
def normalize_labels(labels):
    unique = sorted(np.unique(labels))
    mapping = {old: new for new, old in enumerate(unique)}
    return np.array([mapping[x] for x in labels])


# --------------------------------------------------
# Main
# --------------------------------------------------
def main(args=None):
    parser = argparse.ArgumentParser(description="Graph-Based Clustering using MST")
    parser.add_argument("--data", type=str, required=True, help="Path to iris.csv")
    parser.add_argument("--k", type=int, default=3, help="Number of clusters")
    parser.add_argument("--distance", type=str, default="euclidean",
                        choices=["euclidean", "manhattan", "cosine"],
                        help="Distance metric")
    args = parser.parse_args(args) # Pass args here

    # Load data
    df = pd.read_csv(args.data)

    # Use only numeric columns
    df_numeric = df.select_dtypes(include=[np.number])

    if df_numeric.shape[1] == 0:
        raise ValueError("No numeric feature columns found in the dataset.")

    X = df_numeric.values
    n = len(X)

    # Step 1: Build complete weighted graph
    edges, dist_matrix = build_complete_graph(X, metric=args.distance)

    # Step 2: Construct MST using Kruskal
    mst_edges = kruskal_mst(n, edges)

    # Step 3: Form clusters by removing (k-1) most expensive edges
    mst_labels = mst_clustering(n, mst_edges, args.k)
    mst_labels = normalize_labels(mst_labels)

    # Step 4: K-Means clustering
    kmeans = KMeans(n_clusters=args.k, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X)
    kmeans_labels = normalize_labels(kmeans_labels)

    # Step 5: Save cluster outputs
    mst_output = pd.DataFrame({
        "SampleId": np.arange(1, n + 1),
        "ClusterLabel": mst_labels
    })
    mst_output.to_csv("mst_clusters.csv", index=False)

    kmeans_output = pd.DataFrame({
        "SampleId": np.arange(1, n + 1),
        "ClusterLabel": kmeans_labels
    })
    kmeans_output.to_csv("kmeans_clusters.csv", index=False)

    # Step 6: Silhouette scores
    # Silhouette score requires at least 2 clusters and fewer than n clusters
    if len(np.unique(mst_labels)) > 1 and len(np.unique(mst_labels)) < n:
        mst_silhouette = silhouette_score(X, mst_labels, metric=args.distance)
    else:
        mst_silhouette = float("nan")

    if len(np.unique(kmeans_labels)) > 1 and len(np.unique(kmeans_labels)) < n:
        kmeans_silhouette = silhouette_score(X, kmeans_labels, metric=args.distance)
    else:
        kmeans_silhouette = float("nan")

    # Step 7: Print comparison report
    print("\n--- Clustering Completed ---")
    print(f"Dataset: {args.data}")
    print(f"Distance Metric: {args.distance}")
    print(f"Number of Clusters (k): {args.k}")

    print("\nFiles generated:")
    print("1. mst_clusters.csv")
    print("2. kmeans_clusters.csv")

    print("\nSilhouette Score Comparison:")
    print(f"MST Clustering     : {mst_silhouette:.4f}")
    print(f"K-Means Clustering : {kmeans_silhouette:.4f}")

    print("\nBrief Comparison Note:")
    print(
        "MST clustering can perform better when clusters are irregular, chained, or non-spherical,\n"
        "because it uses graph connectivity instead of assuming compact circular groups.\n"
        "K-Means usually performs better when clusters are spherical, well-separated, and balanced in size.\n"
        "MST clustering may be sensitive to noise and outliers, since one unusual point can affect long edges\n"
        "in the tree. K-Means is usually faster and simpler for large datasets, but it may split curved clusters\n"
        "incorrectly because it relies on centroid distance."
    )

    print("\nWhy MST can capture non-spherical clusters:")
    print(
        "MST-based clustering connects nearby points using minimum total edge cost.\n"
        "It focuses on local neighborhood structure rather than global cluster shape.\n"
        "Because of this, points forming curved, elongated, or chain-like patterns can remain connected.\n"
        "After removing the largest edges, natural gaps in the graph separate clusters.\n"
        "This makes MST suitable for clusters that are not circular or convex.\n"
        "In contrast, K-Means assumes clusters are centered around centroids.\n"
        "That assumption works well for compact round groups but not for irregular shapes.\n"
        "So MST is often better for discovering structure hidden in the connectivity of the data."
    )


if __name__ == "__main__":
    # Example usage for Colab
    # You need to upload an 'iris.csv' file to your Colab environment or adjust the path.
    # For a real run, ensure your data file is accessible.

    # Download iris.csv if not present
    import os
    if not os.path.exists("iris.csv"):
        print("Downloading iris.csv...")
        # Using sklearn to load iris dataset and save it as a CSV
        from sklearn.datasets import load_iris
        iris = load_iris(as_frame=True)
        iris_df = iris.frame
        iris_df.to_csv("iris.csv", index=False)
        print("iris.csv downloaded.")


    mock_args = [
        "--data", "iris.csv",
        "--k", "3",
        "--distance", "euclidean"
    ]
    main(mock_args)


iris.csv downloaded.

--- Clustering Completed ---
Dataset: iris.csv
Distance Metric: euclidean
Number of Clusters (k): 3

Files generated:
1. mst_clusters.csv
2. kmeans_clusters.csv

Silhouette Score Comparison:
MST Clustering     : 0.5782
K-Means Clustering : 0.5819

Brief Comparison Note:
MST clustering can perform better when clusters are irregular, chained, or non-spherical,
because it uses graph connectivity instead of assuming compact circular groups.
K-Means usually performs better when clusters are spherical, well-separated, and balanced in size.
MST clustering may be sensitive to noise and outliers, since one unusual point can affect long edges
in the tree. K-Means is usually faster and simpler for large datasets, but it may split curved clusters
incorrectly because it relies on centroid distance.

Why MST can capture non-spherical clusters:
MST-based clustering connects nearby points using minimum total edge cost.
It focuses on local neighborhood structure rather than global